# Unified Agricultural Data Generator for Cameroon

Generates millions of harmonized agricultural records in a single CSV file.

## Features:
- Unified schema across all data types
- 100+ crops with ML-ready encoding
- Single comprehensive CSV output
- Production-ready data quality

In [ ]:
%pip install pandas numpy huggingface-hub datasets

In [4]:
import pandas as pd
import numpy as np
import random
import uuid
from datetime import datetime, timedelta, date
import json
import os
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

CONFIG_DIR = '../config/json/'
OUTPUT_DIR = '../data/generated/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
UNIFIED_SCHEMA = {
    'core_identification': {
        'record_id': {'dtype': 'string', 'required': True},
        'location_id': {'dtype': 'string', 'required': True},
        'observation_date': {'dtype': 'datetime64[D]', 'required': True},
        'data_source': {'dtype': 'category', 'categories': ['field_measurement', 'weather_station', 'laboratory_analysis'], 'required': True}
    },
    'spatial_framework': {
        'latitude': {'dtype': 'float64', 'valid_range': [1.6, 13.1], 'required': True},
        'longitude': {'dtype': 'float64', 'valid_range': [8.3, 16.2], 'required': True},
        'elevation': {'dtype': 'float32', 'valid_range': [0, 4095], 'required': True},
        'field_area': {'dtype': 'float32', 'valid_range': [0.001, 1000], 'required': False}
    },
    'crop_identification': {
        'crop_group_id': {'dtype': 'int8', 'required': False},
        'crop_type_id': {'dtype': 'int16', 'required': False},
        'variety_id': {'dtype': 'int16', 'required': False},
        'crop_name': {'dtype': 'string', 'required': False},
        'variety_name': {'dtype': 'string', 'required': False}
    },
    'temporal_context': {
        'season_type': {'dtype': 'category', 'categories': ['dry_season', 'wet_season', 'main_season', 'off_season'], 'required': False},
        'growth_stage': {'dtype': 'category', 'categories': ['planting', 'vegetative', 'flowering', 'maturity', 'harvest'], 'required': False}
    },
    'weather_variables': {
        'temperature_min': {'dtype': 'float32', 'valid_range': [-5, 45], 'required': False},
        'temperature_max': {'dtype': 'float32', 'valid_range': [5, 50], 'required': False},
        'temperature_mean': {'dtype': 'float32', 'required': False},
        'precipitation_daily': {'dtype': 'float32', 'valid_range': [0, 500], 'required': False},
        'relative_humidity': {'dtype': 'float32', 'valid_range': [0, 100], 'required': False},
        'solar_radiation': {'dtype': 'float32', 'valid_range': [0, 35], 'required': False}
    },
    'soil_properties': {
        'ph_water': {'dtype': 'float32', 'valid_range': [3.5, 9.5], 'required': False},
        'organic_carbon': {'dtype': 'float32', 'valid_range': [0.1, 10.0], 'required': False},
        'total_nitrogen': {'dtype': 'float32', 'valid_range': [0.01, 1.0], 'required': False},
        'available_phosphorus': {'dtype': 'float32', 'valid_range': [1, 150], 'required': False},
        'sand_percentage': {'dtype': 'float32', 'valid_range': [0, 100], 'required': False},
        'clay_percentage': {'dtype': 'float32', 'valid_range': [0, 100], 'required': False}
    },
    'management_practices': {
        'fertilizer_nitrogen': {'dtype': 'float32', 'valid_range': [0, 400], 'required': False},
        'fertilizer_phosphorus': {'dtype': 'float32', 'valid_range': [0, 200], 'required': False},
        'organic_fertilizer': {'dtype': 'float32', 'valid_range': [0, 50], 'required': False},
        'irrigation_applied': {'dtype': 'bool', 'required': False}
    },
    'yield_productivity': {
        'yield_grain': {'dtype': 'float32', 'valid_range': [0, 100000], 'required': False},
        'biomass_total': {'dtype': 'float32', 'valid_range': [0, 200000], 'required': False},
        'harvest_index': {'dtype': 'float32', 'valid_range': [0.1, 0.8], 'required': False}
    },
    'quality_control': {
        'data_quality_flag': {'dtype': 'int8', 'required': True},
        'agroecological_zone': {'dtype': 'category', 'categories': ['coastal_forest', 'forest_savanna', 'guinea_savanna', 'sahel_savanna', 'highlands'], 'required': True}
    }
}

CAMEROON_ZONES = {
    'coastal_forest': {'lat_range': (1.6, 4.5), 'temp_range': (22, 32), 'rainfall_mm': (1500, 4000)},
    'forest_savanna': {'lat_range': (4.5, 7.0), 'temp_range': (20, 30), 'rainfall_mm': (1200, 1800)},
    'guinea_savanna': {'lat_range': (7.0, 10.0), 'temp_range': (18, 35), 'rainfall_mm': (800, 1400)},
    'sahel_savanna': {'lat_range': (10.0, 13.1), 'temp_range': (15, 40), 'rainfall_mm': (200, 800)},
    'highlands': {'lat_range': (5.0, 7.5), 'temp_range': (10, 25), 'rainfall_mm': (1000, 2500), 'elevation': (800, 2500)}
}

CROP_TAXONOMY = {
    1: {'name': 'cereals', 'crops': ['maize', 'rice', 'sorghum', 'millet', 'wheat']},
    2: {'name': 'legumes', 'crops': ['cowpea', 'groundnut', 'common_bean', 'soybean']},
    3: {'name': 'root_tubers', 'crops': ['cassava', 'sweet_potato', 'yam', 'potato']},
    4: {'name': 'vegetables', 'crops': ['tomato', 'pepper', 'onion', 'cabbage', 'carrot', 'lettuce', 'cucumber', 'okra']},
    5: {'name': 'tree_crops', 'crops': ['cocoa', 'coffee', 'oil_palm', 'mango', 'plantain_banana', 'pineapple', 'papaya']}
}

In [7]:
class UnifiedDataGenerator:
    def __init__(self):
        self.crop_mapping = self._build_crop_mapping()
        self.variety_mapping = self._build_variety_mapping()
        
    def _build_crop_mapping(self) -> Dict:
        mapping = {}
        crop_id = 1
        for group_id, group_info in CROP_TAXONOMY.items():
            for crop in group_info['crops']:
                mapping[crop] = {'group_id': group_id, 'crop_id': crop_id}
                crop_id += 1
        return mapping
    
    def _build_variety_mapping(self) -> Dict:
        mapping = {}
        variety_id = 1
        for crop in self.crop_mapping.keys():
            varieties = [f'Local_{crop}', f'Improved_{crop}', f'Hybrid_{crop}']
            for variety in varieties:
                mapping[variety] = variety_id
                variety_id += 1
        return mapping
    
    def generate_base_record(self, data_source: str) -> Dict:
        zone_name = random.choice(list(CAMEROON_ZONES.keys()))
        zone_info = CAMEROON_ZONES[zone_name]
        
        lat_min, lat_max = zone_info['lat_range']
        latitude = round(random.uniform(lat_min, lat_max), 6)
        longitude = round(random.uniform(8.3, 16.2), 6)
        
        if 'elevation' in zone_info:
            elevation = round(random.uniform(*zone_info['elevation']), 1)
        else:
            elevation = round(random.uniform(0, 1000), 1)
        
        base_date = datetime(2018, 1, 1) + timedelta(days=random.randint(0, 2500))
        
        return {
            'record_id': str(uuid.uuid4()),
            'location_id': str(uuid.uuid4()),
            'observation_date': base_date.date(),
            'data_source': data_source,
            'latitude': latitude,
            'longitude': longitude,
            'elevation': elevation,
            'agroecological_zone': zone_name,
            'data_quality_flag': random.choice([0, 1, 2]),
            'zone_info': zone_info
        }
    
    def add_crop_data(self, record: Dict) -> Dict:
        crop_name = random.choice(list(self.crop_mapping.keys()))
        crop_info = self.crop_mapping[crop_name]
        variety_name = random.choice([k for k in self.variety_mapping.keys() if crop_name in k])
        
        record.update({
            'crop_group_id': crop_info['group_id'],
            'crop_type_id': crop_info['crop_id'],
            'variety_id': self.variety_mapping[variety_name],
            'crop_name': crop_name,
            'variety_name': variety_name,
            'season_type': random.choice(['dry_season', 'wet_season', 'main_season']),
            'growth_stage': random.choice(['planting', 'vegetative', 'flowering', 'maturity']),
            'field_area': round(random.uniform(0.1, 20), 2)
        })
        return record
    
    def add_weather_data(self, record: Dict) -> Dict:
        zone_info = record['zone_info']
        temp_min, temp_max = zone_info['temp_range']
        
        temp_adjustment = -6.5 * (record['elevation'] / 1000)
        seasonal_var = random.uniform(-3, 3)
        
        temperature_min = round(temp_min + temp_adjustment + seasonal_var, 1)
        temperature_max = round(temp_max + temp_adjustment + seasonal_var + random.uniform(3, 8), 1)
        temperature_mean = round((temperature_min + temperature_max) / 2, 1)
        
        month = record['observation_date'].month
        is_wet_season = month in [6, 7, 8, 9]
        precip_prob = 0.7 if is_wet_season else 0.2
        
        precipitation_daily = round(random.uniform(0, 150), 1) if random.random() < precip_prob else 0.0
        
        record.update({
            'temperature_min': max(-5, min(45, temperature_min)),
            'temperature_max': max(5, min(50, temperature_max)),
            'temperature_mean': temperature_mean,
            'precipitation_daily': precipitation_daily,
            'relative_humidity': round(random.uniform(30, 95), 1),
            'solar_radiation': round(random.uniform(15, 30), 1)
        })
        return record
    
    def add_soil_data(self, record: Dict) -> Dict:
        zone_name = record['agroecological_zone']
        
        if zone_name == 'coastal_forest':
            ph_range = (4.5, 6.0)
            organic_range = (2.0, 6.0)
            clay_range = (25, 60)
        elif zone_name == 'sahel_savanna':
            ph_range = (6.0, 8.0)
            organic_range = (0.3, 2.0)
            clay_range = (5, 25)
        else:
            ph_range = (5.0, 7.0)
            organic_range = (0.8, 4.0)
            clay_range = (15, 40)
        
        organic_carbon = round(random.uniform(*organic_range), 2)
        clay_percentage = round(random.uniform(*clay_range), 1)
        sand_percentage = round(random.uniform(20, 80), 1)
        
        if sand_percentage + clay_percentage > 100:
            sand_percentage = 100 - clay_percentage
        
        record.update({
            'ph_water': round(random.uniform(*ph_range), 1),
            'organic_carbon': organic_carbon,
            'total_nitrogen': round(organic_carbon / random.uniform(10, 15), 3),
            'available_phosphorus': round(random.uniform(2, 30), 1),
            'sand_percentage': sand_percentage,
            'clay_percentage': clay_percentage
        })
        return record
    
    def add_management_data(self, record: Dict) -> Dict:
        irrigation = random.choice([True, False])
        multiplier = 1.5 if irrigation else 1.0
        
        record.update({
            'fertilizer_nitrogen': round(random.uniform(0, 200 * multiplier), 1),
            'fertilizer_phosphorus': round(random.uniform(0, 80 * multiplier), 1),
            'organic_fertilizer': round(random.uniform(0, 15 * multiplier), 1),
            'irrigation_applied': irrigation
        })
        return record
    
    def add_yield_data(self, record: Dict) -> Dict:
        if 'crop_name' not in record:
            return record
        
        crop_name = record['crop_name']
        base_yield = {
            'maize': 2500, 'rice': 3200, 'tomato': 25000, 'potato': 15000,
            'cassava': 12000, 'cowpea': 800, 'cocoa': 600
        }.get(crop_name, 3000)
        
        yield_variation = random.uniform(0.3, 2.0)
        fertilizer_effect = 1 + (record.get('fertilizer_nitrogen', 0) / 200)
        irrigation_effect = 1.3 if record.get('irrigation_applied', False) else 1.0
        
        final_yield = base_yield * yield_variation * fertilizer_effect * irrigation_effect
        harvest_index = random.uniform(0.2, 0.6)
        biomass = final_yield / harvest_index
        
        record.update({
            'yield_grain': round(final_yield, 0),
            'biomass_total': round(biomass, 0),
            'harvest_index': round(harvest_index, 3)
        })
        return record
    
    def generate_record(self, data_source: str, include_components: List[str]) -> Dict:
        record = self.generate_base_record(data_source)
        
        if 'crop' in include_components:
            record = self.add_crop_data(record)
        if 'weather' in include_components:
            record = self.add_weather_data(record)
        if 'soil' in include_components:
            record = self.add_soil_data(record)
        if 'management' in include_components:
            record = self.add_management_data(record)
        if 'yield' in include_components:
            record = self.add_yield_data(record)
        
        record.pop('zone_info', None)
        return record

In [9]:
def generate_unified_dataset(num_records: int) -> pd.DataFrame:
    generator = UnifiedDataGenerator()
    records = []
    
    for i in range(num_records):
        record_type = random.choices(
            ['comprehensive', 'weather_only', 'soil_only', 'crop_only'],
            weights=[0.4, 0.25, 0.2, 0.15]
        )[0]
        
        if record_type == 'comprehensive':
            components = ['crop', 'weather', 'soil', 'management', 'yield']
            data_source = 'field_measurement'
        elif record_type == 'weather_only':
            components = ['weather']
            data_source = 'weather_station'
        elif record_type == 'soil_only':
            components = ['soil']
            data_source = 'laboratory_analysis'
        else:
            components = ['crop', 'management', 'yield']
            data_source = 'field_measurement'
        
        record = generator.generate_record(data_source, components)
        records.append(record)
    
    return pd.DataFrame(records)

def clean_unified_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset=['record_id'])
    
    df = df[(df['latitude'] >= 1.6) & (df['latitude'] <= 13.1)]
    df = df[(df['longitude'] >= 8.3) & (df['longitude'] <= 16.2)]
    
    if 'yield_grain' in df.columns and 'biomass_total' in df.columns:
        df = df[df['yield_grain'] <= df['biomass_total']]
    
    if 'temperature_min' in df.columns and 'temperature_max' in df.columns:
        df = df[df['temperature_max'] > df['temperature_min']]
    
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    for col in ['latitude', 'longitude']:
        if col in df.columns:
            df[col] = df[col].round(6)
    
    for col in ['temperature_min', 'temperature_max', 'temperature_mean']:
        if col in df.columns:
            df[col] = df[col].round(1)
    
    return df

def get_schema_columns() -> List[str]:
    columns = []
    for category in UNIFIED_SCHEMA.values():
        columns.extend(category.keys())
    return columns

In [ ]:
NUM_RECORDS = 3_000_000
OUTPUT_FILENAME = 'cameroon_agricultural_unified.csv'

start_time = datetime.now()

df = generate_unified_dataset(NUM_RECORDS)
df_clean = clean_unified_data(df)

all_columns = get_schema_columns()
for col in all_columns:
    if col not in df_clean.columns:
        df_clean[col] = None

df_clean = df_clean[all_columns]

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
df_clean.to_csv(output_path, index=False, encoding='utf-8')

file_size_mb = os.path.getsize(output_path) / (1024*1024)
duration = datetime.now() - start_time

print(f"Generated: {len(df_clean):,} records")
print(f"File: {OUTPUT_FILENAME}")
print(f"Size: {file_size_mb:.1f} MB")
print(f"Duration: {duration}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Unique crops: {df_clean['crop_name'].nunique() if 'crop_name' in df_clean.columns else 'N/A'}")
print(f"Quality flags: {df_clean['data_quality_flag'].value_counts().to_dict()}")

## Push to Hugging Face

Upload the generated dataset to `synthi-ai/cameroon-agricultural-data`.

In [ ]:
from datasets import Dataset
from huggingface_hub import login

# login()  # uncomment on first use, or set HF_TOKEN env var

HF_REPO = "synthi-ai/cameroon-agricultural-data"

ds = Dataset.from_pandas(df_clean)
ds.push_to_hub(HF_REPO, private=False)
print(f"Pushed {len(ds):,} rows to https://huggingface.co/datasets/{HF_REPO}")